# 🛠️ Active Learning Workshop: Implementing an Inverted Matrix (Jupyter + GitHub Edition)
## 🔍 Workshop Theme
*Readable, correct, and collaboratively reviewed code—just like in the real world.*


Welcome to the 90-minute workshop! In this hands-on session, your team will build an **Inverted Index** pipeline, the foundation of many intelligent systems that need fast and relevant access to text data — such as AI agents.

### 👥 Team Guidelines
- Work in teams of 3.
- Submit one completed Jupyter Notebook per team.
- The final notebook must contain **Markdown explanations** and **Python code**.
- Push your notebook to GitHub and share the `.git` link before class ends.

---
## 🔧 Workshop Tasks Overview

1. **Document Collection**
2. **Tokenizer Implementation**
3. **Normalization Pipeline (Stemming, Stop Words, etc.)**
4. **Build and Query the Inverted Index**

> Each step includes a sample **talking point**. Your team must add your own custom **Markdown + code cells** with a **second talking point**, and test your Inverted Index with **2 phrase queries**.




## 🧠 Learning Objectives
- Implement an **Inverted Matrix** using real-world data during the NLP process.
- Build **Jupyter Notebooks** with well-structured code and clear Markdown documentation.
- Use **Git and GitHub** for collaborative version control and code sharing.
- Identify and articulate coding issues ("**talking points**") and insert them directly into peer notebooks.
- Practice **collaborative debugging**, professional peer feedback, and improve code quality.

## 🧩 Workshop Structure (120 Minutes)
1. **Instructor Use Case Introduction** *(15 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(45 min)* – Complete all challenges in the notebook. Work as teams but **make sure every individual has their own copy of the notebook with unique algorithms and queries**.
3. **Push to GitHub** *(15 min)* – Individuals commit and push finished notebooks. **Make sure to include your names and the team you worked with so it is easy to identify the team that developed the code**.
4. **Instructor Review** *(30 min)* - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**, assisting with individual work.
5. **Email Delivery** *(15 min)* – Each team sends the instructor an email **with the *.git link to the GitHub repo of every team member (one email/team)**. Subject on the email is: PROG8245 - Inverted Matrix  Workshop, Team #_____.
6. The GitHub will be reviewed by the instructor and feedback will be provided if deemed necessary. Work will be assessed during the workshop.


## 💻 Submission Checklist
- ✅ `IR_InvertedMatrix_Workshop.ipynb` with:
  - Demo code: Document Collection, Tokenizer, Normalization Pipeline, and challenges coded and solved.
  - Markdown explanations for each major step
  - **Labeled talking point(s)** and 2 phrase query tests
- ✅ `README.md` with:
  - Dataset description
  - Team member names
  - Link to the dataset and license (if public)
- ✅ GitHub Repo:
  - Public repo named `IR-invertedmatrix-workshop`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - At least **one commit containing one meaningful talking point**

## 📄 Step 1: Document Collection


### 🗣 Instructor Talking Point:
> We begin by gathering a text corpus. To build a robust index, your vocabulary should include **over 2000 unique words**. You can use scraped articles, academic papers, or open datasets.

### 🔧 Your Task:
- Collect at least 20+ text documents.
- Ensure the vocabulary exceeds 2000 unique words.
- Load the documents into a list for processing.


In [ ]:
import os
import requests
import feedparser

# Example: Download blog posts from a public RSS feed and save as .txt files
def fetch_and_save_blog_posts(rss_url, save_folder, max_posts=20):
    feed = feedparser.parse(rss_url)
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    count = 0
    for entry in feed.entries:
        if count >= max_posts:
            break
        title = entry.get('title', 'untitled')
        content = entry.get('summary', '')
        # Clean filename
        filename = f"blog_{count+1}.txt"
        filepath = os.path.join(save_folder, filename)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"{title}\n\n{content}")
        count += 1
    print(f"Saved {count} blog posts to {save_folder}")

# Example RSS feed: Python Software Foundation blog
rss_url = 'https://pyfound.blogspot.com/feeds/posts/default?alt=rss'
fetch_and_save_blog_posts(rss_url, 'sample_docs', max_posts=20)

In [ ]:
# Example: Load text files from a folder
import os

def load_documents(folder_path):
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as file:
                documents.append(file.read())
    return documents

# Replace 'sample_docs/' with your actual folder
documents = load_documents('sample_docs/')
print(f"Loaded {len(documents)} documents.")


## ✂️ Step 2: Tokenizer


### 🗣 Instructor Talking Point:
> The tokenizer breaks raw text into a stream of words (tokens). This is the foundation for every later step in IR and NLP.

### 🔧 Your Task:
- Implement a basic tokenizer that splits text into lowercase words.
- Handle punctuation removal and basic non-alphanumeric filtering.


In [ ]:
import re

def tokenize(text):
    """
    Tokenize raw text into a list of lowercase word tokens.
    Steps:
      1. Lowercase the entire string.
      2. Replace any non-alphanumeric character (punctuation, digits, symbols)
         with a space so word boundaries are preserved.
      3. Split on whitespace and discard empty strings.
    """
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation / non-alpha characters
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 3. Split and filter empty tokens
    tokens = [t for t in text.split() if t]
    return tokens

# Test on one document
tokens = tokenize(documents[0])
print(f"Total tokens in doc 0: {len(tokens)}")
print("First 20 tokens:", tokens[:20])


### 💬 Team Talking Point — Tokenizer Design

> **Why does tokenizer quality matter downstream?**

A naive tokenizer that splits only on whitespace leaves punctuation attached to words (e.g., `"Python,"` vs `"Python"`), causing them to be indexed as **different terms**. This inflates the vocabulary, reduces recall, and breaks phrase queries. By stripping non-alpha characters with a regex *before* splitting, we ensure `"Python,"`, `"Python."`, and `"Python"` all map to the same token `"python"`.

**Trade-off:** Stripping digits means numeric identifiers like `"Python3"` or `"GPT-4"` lose their numeral. Teams should decide whether their domain requires digit-aware tokenisation.

## 🔁 Step 3: Normalization Pipeline (Stemming, Stop Word Removal, etc.)


### 🗣 Instructor Talking Point:
> Now we normalize tokens: convert to lowercase, remove stop words, apply stemming or affix stripping. This reduces redundancy and enhances search accuracy.

### 🔧 Your Task:
- Use `nltk` to remove stopwords and apply stemming.


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def normalize_tokens(tokens):
    return [stemmer.stem(t) for t in tokens if t not in stop_words]

# Example: normalize one document
norm_tokens = normalize_tokens(tokens)
print(norm_tokens[:20])


### 💬 Team Talking Point — Stemming vs. Lemmatisation

> **Should we use Porter Stemming or WordNet Lemmatisation?**

Porter Stemming is a fast rule-based affix-stripping algorithm, but can be aggressive: `"university"` becomes `"univers"`. Lemmatisation returns real dictionary forms using a lexical database (`"running"` -> `"run"`), but requires POS tagging and is slower.

**Recommendation:** For this workshop corpus, Porter Stemming provides a good speed/accuracy balance. For production systems where interpretability matters, prefer lemmatisation.

## 🔍 Step 4: Inverted Index


### 🗣 Instructor Talking Point:
> We now map each normalized token to the list of document IDs in which it appears. This is the core structure that allows fast Boolean and phrase queries.

### 🔧 Your Task:
- Build the inverted index using a dictionary.
- Add code to support phrase queries using positional indexing.


In [ ]:
from collections import defaultdict

def build_inverted_index(documents):
    """
    Build a basic inverted index.
    Structure: { term -> set(doc_ids) }

    For each document:
      1. Tokenise and normalise the text.
      2. For every normalised token, record the document ID.
    """
    index = defaultdict(set)
    for doc_id, text in enumerate(documents):
        # Full pipeline: tokenise then normalise
        norm_tokens = normalize_tokens(tokenize(text))
        for token in norm_tokens:
            index[token].add(doc_id)
    return index

inverted_index = build_inverted_index(documents)

# Convert sets to sorted lists for readable output
print("Inverted index (first 10 terms):")
for term, doc_ids in list(inverted_index.items())[:10]:
    print(f"  '{term}': {sorted(doc_ids)}")

print(f"\nTotal unique terms in index: {len(inverted_index)}")


### 💬 Team Talking Point — Boolean vs. Ranked Retrieval

> **Our inverted index supports exact Boolean queries — is that enough?**

A classic inverted index returns the *set* of documents containing a term. Boolean queries are fast but return unordered results with no relevance ranking. A **ranked retrieval model** (e.g., BM25 or TF-IDF cosine similarity) scores and orders documents by relevance, greatly improving usability over large corpora.

The inverted index built here is the **shared foundation** for both approaches: ranked retrieval simply augments each posting with term frequency or positional data, which we implement in the Additional Challenges below.

## 🧪 Test: Phrase Queries


### 🗣 Instructor Talking Point:
> A phrase query requires the exact sequence of terms (e.g., "machine learning"). To support this, extend the inverted index to store positions, not just docIDs.

### 🔧 Your Task:
- Implement 2 phrase queries **using the inverted matrix**.
- Demonstrate that they return the correct documents.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Phrase Query using a Positional Inverted Index
# Strategy:
#   1. Build a positional index (term -> {doc_id -> [positions]}).
#   2. For a multi-word phrase, find documents that contain ALL terms.
#   3. Verify that for at least one starting position in doc, each subsequent
#      term appears exactly one position later (consecutive positions).
# ─────────────────────────────────────────────────────────────────────────────

from collections import defaultdict

def build_positional_index_for_query(documents):
    """Positional index: term -> {doc_id: [pos, ...]}"""
    pos_index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        norm_tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(norm_tokens):
            pos_index[token][doc_id].append(pos)
    return pos_index

_phrase_pos_index = build_positional_index_for_query(documents)

def phrase_query(query, documents, pos_index=None):
    """
    Return a list of doc_ids that contain the exact phrase.

    Algorithm (positional merge):
      - Normalise the query terms.
      - Candidate docs = intersection of posting lists for all terms.
      - For each candidate doc, check whether any sequence of positions
        forms consecutive offsets (p, p+1, p+2, …).
    """
    if pos_index is None:
        pos_index = _phrase_pos_index

    # Normalise query the same way as documents
    query_terms = normalize_tokens(tokenize(query))
    if not query_terms:
        return []

    # Candidate documents must contain every query term
    candidate_docs = set(pos_index[query_terms[0]].keys())
    for term in query_terms[1:]:
        candidate_docs &= set(pos_index[term].keys())

    results = []
    for doc_id in sorted(candidate_docs):
        # Positions of the first term in this doc
        start_positions = pos_index[query_terms[0]][doc_id]
        for start_pos in start_positions:
            # Check if every subsequent term appears at start_pos + offset
            match = all(
                (start_pos + offset) in pos_index[term][doc_id]
                for offset, term in enumerate(query_terms[1:], start=1)
            )
            if match:
                results.append(doc_id)
                break  # Found phrase in this doc, move to next
    return results


# ── Two example phrase queries (adjust to terms present in your corpus) ──
query1 = "open source"
query2 = "python software"

results1 = phrase_query(query1, documents)
results2 = phrase_query(query2, documents)

print(f"Documents containing the phrase '{query1}': {results1}")
print(f"Documents containing the phrase '{query2}': {results2}")


## 🧠 Additional Challenge: Use Positional Indexes to compare TF and IDF

Implement Positional Indexes, an advanced version of an inverted index that not only stores which documents a term appears in, but also where (at what positions) the term occurs within each document.

Apply it to the documents you used to produce the Inverted Matrix during the Active Learning Workshop.

Then compare it against a **Term Document Count Matrix** (which you don't have to implement)

And finally, fill in the blanks in the table below.

| Term | What is it? | How is it used? | Pros | Cons |
|------|-------------|-----------------|------|------|
| **Term Frequency (TF)** | A count of how many times a term `t` appears in a specific document `d`. It measures the **local importance** of a term within one document. Formula: `TF(t, d) = count of t in d` | Used as a component of the TF-IDF score to weight terms that appear frequently in a document. Also used directly in ranked retrieval to promote documents where the query term appears often. Combined with the positional index, TF can be derived by counting the length of the position list for a term in a document. | Simple and fast to compute. Directly reflects how central a term is to a particular document. Naturally boosts documents where the query word is a main topic. | Biased toward long documents (a 10,000-word doc will always have higher raw TF than a 500-word doc for the same term). Ignores whether the term is meaningful across the corpus — common words like *"the"* get inflated TF scores even after stop-word removal. Needs normalisation (e.g., divide by document length) to be fair. |
| **Inverse Document Frequency (IDF weight)** | A logarithmic measure of how **rare** a term is across the entire corpus of `N` documents. Formula: `IDF(t) = log(N / n_t)` where `n_t` is the number of documents containing term `t`. High IDF = rare term; Low IDF = common term. | Multiplied with TF to form the TF-IDF score: `TF-IDF(t, d) = TF(t, d) × IDF(t)`. This down-weights terms that appear in many documents (e.g., *"python"* in a Python blog corpus) and up-weights terms unique to few documents, improving search precision and document ranking. Derived from the positional index by counting the number of distinct doc IDs per term. | Elegantly penalises corpus-wide common words without needing a manual stop-word list. Balances TF so that rare, meaningful terms drive relevance. Language-agnostic and easy to compute from any inverted or positional index. | Assumes term independence (ignores co-occurrence). Sensitive to corpus size — IDF scores shift if the corpus grows or shrinks. Does not capture semantic meaning (synonyms get separate IDF scores). A term appearing in every document gets `IDF = log(1) = 0`, making it invisible even if contextually relevant. |

---

### 💬 Talking Point: Which implementation is preferred for searching bigrams (biwords)?

**Answer: The Positional Index is strongly preferred for bigram / phrase search.**

| Criterion | Positional Index | TF-IDF / Term-Doc Count Matrix |
|-----------|-----------------|--------------------------------|
| Can verify two terms are *adjacent*? | **Yes** — stores exact positions, so `pos(term2) = pos(term1) + 1` can be checked directly | **No** — only stores counts; cannot distinguish `"machine ... learning"` (far apart) from `"machine learning"` (consecutive) |
| False positives for bigram queries | None — positional merge is exact | High — both terms in doc does *not* mean they appear together |
| Memory cost | Higher — O(total tokens) position entries | Lower — O(unique terms × docs) |
| Query speed | O(posting list size) merge | Would require full document scan to verify adjacency |
| Supports proximity queries (within N words)? | Yes | No |

**Why the positional index wins for bigrams:** A bigram query like *"open source"* requires that the two tokens appear **consecutively** in the document. The positional index stores `[pos_0, pos_1, ...]` for each `(term, doc_id)` pair, so the algorithm simply checks whether any position `p` for *"open"* satisfies `p+1 ∈ positions("sourc")` in the same document. This is an O(|postings|) merge — fast and exact.

A TF-IDF matrix or plain term-document count matrix only knows that both stemmed terms appear *somewhere* in the document. Confirming adjacency would require loading and re-scanning the raw document text, eliminating the speed benefit of having an index at all. TF-IDF is powerful for **ranking** documents by relevance but is not a substitute for positional data when **phrase precision** is required.

#### Sample solution:

In [ ]:
# Implement Positional Indexes: Store term positions for each document
from collections import defaultdict

def build_positional_index(documents):
    positional_index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(tokens):
            positional_index[token][doc_id].append(pos)
    return positional_index

# Build positional index for the loaded documents
positional_index = build_positional_index(documents)

# Example: Show positions for a sample term
sample_term = 'python'  # Change to any term you want to inspect
print(f"Positions for term '{sample_term}':")
for doc_id, positions in positional_index[sample_term].items():
    print(f"  Document {doc_id}: {positions}")

## 🧠 Additional Challenge 2: Implement Optimized Positional Indexes

Find a way to method to optimize the memory management to boost code performance for very large documents.
Implement the new code in the space below:

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Challenge 2: Memory-Efficient Positional Index
#
# Optimisations applied:
#   1. Use array.array('I', …) instead of plain Python lists for position
#      lists – each element costs 4 bytes instead of 28+ bytes (Python int).
#   2. Use a single flat dict with tuple keys (term, doc_id) as an alternative
#      to nested defaultdicts, reducing Python object overhead.
#   3. Lazy sorting – positions are appended in order so no sort is needed.
#   4. Term interning – reuse the same string object for repeated terms via
#      sys.intern(), avoiding duplicate string allocations.
# ─────────────────────────────────────────────────────────────────────────────

import sys
import array
from collections import defaultdict

def build_positional_index(documents):
    """
    Memory-efficient positional index.
    Returns: dict[(term, doc_id)] -> array.array('I', [positions])

    Compared to a naive nested defaultdict(list):
      - array.array uses ~7x less memory per position entry.
      - Flat dict avoids inner defaultdict objects (saves ~200 bytes/term).
      - sys.intern() shares string objects across the index.
    """
    pos_index: dict = {}          # (term, doc_id) -> array.array('I')

    for doc_id, text in enumerate(documents):
        norm_tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(norm_tokens):
            token = sys.intern(token)       # reuse string object
            key   = (token, doc_id)
            if key not in pos_index:
                pos_index[key] = array.array('I')  # unsigned int, 4 bytes each
            pos_index[key].append(pos)

    return pos_index


# Build optimised positional index
positional_index = build_positional_index(documents)

# ── Memory comparison ──────────────────────────────────────────────────────
import sys

total_bytes = sum(
    sys.getsizeof(v) for v in positional_index.values()
)
print(f"Optimised positional index: {len(positional_index):,} (term, doc) pairs")
print(f"Total memory for position arrays: {total_bytes / 1024:.1f} KB")

# ── Lookup example ─────────────────────────────────────────────────────────
sample_term = sys.intern('python')   # must intern for key lookup
print(f"\nPositions for term '{sample_term}':")
for (term, doc_id), positions in positional_index.items():
    if term == sample_term:
        print(f"  Document {doc_id}: {list(positions)}")


### 📊 Comparison: Positional Index vs. Term Document Count Matrix

A **Positional Index** stores not only which documents a term appears in, but also the exact positions of each term within those documents. In contrast, a **Term Document Count Matrix** (TDCM) simply records the frequency of each term in each document, without any positional information.

| Feature                      | Positional Index                | Term Document Count Matrix (TDCM) |
|------------------------------|---------------------------------|-----------------------------------|
| Stores term positions        | Yes                             | No                                |
| Supports phrase/bigram search| Yes                             | No                                |
| Memory usage                 | Higher                          | Lower                             |
| Query speed for phrases      | Fast                            | Slow (requires scanning)          |
| Useful for                   | Phrase queries, proximity search | Keyword frequency analysis         |
| Implementation complexity    | Higher                          | Lower                             |

**Summary:**
- Use a positional index for advanced search features like phrase and proximity queries.
- Use a TDCM for simple keyword frequency analysis and ranking.

## 🧠 Additional Challenge 3: Implement the Inverse Document Frequency (IDF). 
Implement the solution in the space below.

### 💬 Team Talking Point — Bigrams: Positional Index vs. TF-IDF

| Criterion | Positional Index | TF-IDF Matrix |
|-----------|-----------------|---------------|
| Supports exact phrase / bigram search | Yes | No |
| Stores term proximity | Yes | No |
| Memory per term | Higher (position list) | Lower (single count) |
| Query speed for phrases | Fast (positional merge) | Slow (full scan) |

**Preferred for bigrams: Positional Index.** A bigram query like *'open source'* requires verifying consecutive positions. The positional index stores exact offsets enabling fast, precise merges. A TF-IDF matrix can only confirm both words appear *somewhere* in a document but cannot confirm adjacency, producing false positives for phrase queries.

In [ ]:
import math

def compute_idf_from_positional_index(positional_index, total_docs):
    """
    Compute Inverse Document Frequency (IDF) for every term.

    Formula:
        IDF(t) = log( N / n_t )

    where:
        N    = total number of documents in the corpus
        n_t  = number of documents that contain term t

    The positional index maps (term, doc_id) -> positions, so
    the number of unique doc_ids for each term equals n_t.
    """
    # Count how many distinct documents each term appears in
    doc_freq: dict[str, set] = {}
    for (term, doc_id) in positional_index.keys():
        doc_freq.setdefault(term, set()).add(doc_id)

    # Compute IDF using natural logarithm
    idf_scores = {
        term: math.log(total_docs / len(doc_ids))
        for term, doc_ids in doc_freq.items()
        if len(doc_ids) > 0   # guard against division by zero
    }
    return idf_scores


# Compute IDF for all terms
idf_scores = compute_idf_from_positional_index(positional_index, len(documents))

print(f"IDF computed for {len(idf_scores):,} unique terms.")

# Show IDF for the sample term (stemmed form)
sample_term = 'softwar'
print(f"IDF for term '{sample_term}': {idf_scores.get(sample_term, 0.0):.4f}")

# Show top-10 highest IDF terms (rarest)
top_rare = sorted(idf_scores.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop-10 rarest terms (highest IDF):")
for term, score in top_rare:
    print(f"  {term:<20} IDF = {score:.4f}")


### 📐 Mathematical Explanation: Inverse Document Frequency (IDF)

The **Inverse Document Frequency (IDF)** is a statistical measure used to evaluate how important a word is to a document in a collection or corpus. The intuition is that terms that appear in many documents are less informative than those that appear in few.

The IDF for a term $t$ is defined as:

$$
\text{IDF}(t) = \log\left(\frac{N}{n_t}\right)
$$

Where:
- $N$ is the total number of documents in the corpus.
- $n_t$ is the number of documents containing the term $t$.

---

#### Example Calculation

Suppose:
- $N = 20$ (total documents)
- $n_{\text{softwar}} = 15$ (documents containing the term 'softwar')

Then:

$$
\text{IDF}(\text{softwar}) = \log\left(\frac{20}{15}\right) \approx 0.2877
$$

So, the IDF for term 'softwar' is $0.2877$.

A higher IDF value means the term is rare across the corpus, while a lower value means the term is common. IDF is often used in combination with Term Frequency (TF) to compute the TF-IDF score:

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$

Where $\text{TF}(t, d)$ is the frequency of term $t$ in document $d$.

This weighting helps highlight terms that are both frequent in a document and rare across the corpus, improving search relevance and document ranking.

## 🧠 Additional Challenge 4: Implement the Term Frequency (TF). 
Implement the solution in the space below.

In [ ]:
import math

term = 'softwar'   # stemmed form of 'software'

def compute_tf(term, document):
    """
    Compute raw Term Frequency (TF) for a single document.

    TF(t, d) = number of times term t appears in document d
               after applying the same tokenise+normalise pipeline.

    Returns:
        int  – raw count of the term in the document.
    """
    norm_tokens = normalize_tokens(tokenize(document))
    tf = norm_tokens.count(term)
    return tf


# ── Display TF for every document ─────────────────────────────────────────
print(f"Term Frequency (TF) for '{term}' across all documents:\n")
print(f"{'Doc ID':<8} {'TF':>6}   {'IDF':>8}   {'TF-IDF':>10}")
print("-" * 40)

idf_val = idf_scores.get(term, 0.0)

for doc_id, doc_text in enumerate(documents):
    tf_val     = compute_tf(term, doc_text)
    tfidf_val  = tf_val * idf_val
    print(f"{doc_id:<8} {tf_val:>6}   {idf_val:>8.4f}   {tfidf_val:>10.4f}")

print()
print(f"IDF('{term}') = {idf_val:.4f}")
tf_doc0 = compute_tf(term, documents[0])
print(f"TF('{term}', doc 0) = {tf_doc0}")
print(f"TF-IDF('{term}', doc 0) = {tf_doc0} × {idf_val:.4f} = {tf_doc0 * idf_val:.4f}")


#### 📊 Example: Term Frequency (TF) and TF-IDF Calculation for 'softwar' in Document 0

TF for term 'softwar' in Document 0:

$$
\text{TF}(\text{softwar}, 0) = n_{\text{softwar}, 0}
$$

TF-IDF for term 'softwar' in Document 0:

$$
\text{TF-IDF}(\text{softwar}, 0) = \text{TF}(\text{softwar}, 0) \times \text{IDF}(\text{softwar}) = n_{\text{softwar}, 0} \times \text{IDF}(\text{softwar})
$$

Where:
- $n_{\text{softwar}, 0}$ is the number of times 'softwar' appears in Document 0.
- $\text{IDF}(\text{softwar})$ is the previously calculated IDF value for 'softwar'.

Substitute the actual values from your output to see the final TF-IDF score for Document 0.